In [0]:
%sql
-- ΕΝΟΤΗΤΑ 1: Αρχικοποίηση Περιβάλλοντος και Δημιουργία Σχημάτων (Medallion Setup)
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "bronze_marios_urban";
CREATE WIDGET TEXT silver_schema DEFAULT "silver_marios_urban";
CREATE WIDGET TEXT gold_schema DEFAULT "gold_marios_urban";
CREATE WIDGET TEXT volume_landing_path DEFAULT "/Volumes/bootcamp/bronze_marios_urban/landing";

In [0]:
%sql
-- Βήμα 3.1: Δημιουργία Quarantine Table (etl_rejects) στο Silver Schema
USE SCHEMA IDENTIFIER(:silver_schema);
CREATE OR REPLACE TABLE etl_rejects(
    source_table STRING,
    reject_reason STRING,
    rejected_record STRING,
    rejected_at TIMESTAMP
);

In [0]:
%sql
-- Βήμα 3.2: Αποθήκευση εγγραφών με κενό email στο etl_rejects
USE SCHEMA IDENTIFIER(:silver_schema);

INSERT INTO etl_rejects (
    source_table, 
    reject_reason, 
    rejected_record, 
    rejected_at)
SELECT 
  'brz_customers' AS source_table,
  'MISSING_EMAIL' AS reject_reason,
  concat_ws(', ', customer_id, first_name, last_name, city, region) AS rejected_record,
  current_timestamp() AS rejected_at
FROM bootcamp.bronze_marios_urban.brz_customers
WHERE email IS NULL OR trim(email) = '';

In [0]:
%sql
-- Βήμα 3.3: Αποθήκευση διπλότυπων πελατών στο etl_rejects
USE SCHEMA IDENTIFIER(:silver_schema);

WITH ranked_customers AS (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    city,
    region,
    row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at ASC) AS rn
  FROM bootcamp.bronze_marios_urban.brz_customers
)
INSERT INTO etl_rejects (source_table, reject_reason, rejected_record, rejected_at)
SELECT 
  'brz_customers' AS source_table,
  'DUPLICATE_CUSTOMER' AS reject_reason,
  concat_ws(', ', customer_id, first_name, last_name, email, city, region) AS rejected_record,
  current_timestamp() AS rejected_at
FROM ranked_customers
WHERE rn > 1;

In [0]:
%sql
--  Βήμα 3.4: Δημιουργία του slv_customers
--Standardise text: trim() στα κείμενα, lower() στο email, upper() στο gender.
--De-duplicate: ROW_NUMBER() για να κρατήσω μόνο 1 εγγραφή ανά customer_id.
--Cast types: μετατροπή birth_date και signup_date σε DATE.
--Data Quality: εξαιρώ τα κενά emails (έχουν πάει ήδη στο etl_rejects).


CREATE OR REPLACE TABLE bootcamp.silver_marios_urban.slv_customers AS
WITH deduped_customers AS (
  SELECT 
    trim(customer_id) AS customer_id,
    trim(first_name) AS first_name,
    trim(last_name) AS last_name,
    lower(trim(email)) AS email,
    upper(trim(gender)) AS gender,
    cast(birth_date AS DATE) AS birth_date,
    trim(city) AS city,
    trim(region) AS region,
    trim(segment) AS segment,
    cast(signup_date AS DATE) AS signup_date,
    row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at ASC) AS rn
  FROM bootcamp.bronze_marios_urban.brz_customers
  WHERE email IS NOT NULL AND trim(email) != ''
)
SELECT 
  customer_id,
  first_name,
  last_name,
  email,
  gender,
  birth_date,
  city,
  region,
  segment,
  signup_date,
  current_timestamp() AS _processed_at
FROM deduped_customers
WHERE rn = 1;

-- Έλεγχος: Περιμένω 485 καθαρούς πελάτες (508 αρχικοί - 8 διπλότυπα - 15 κενά emails)
SELECT count(*) AS total_customers FROM bootcamp.silver_marios_urban.slv_customers;

In [0]:
%sql
--Βήμα 3.5:Έλεγχος για τυχόν διπλότυπα product_id.
--Αν το αποτέλεσμα είναι κενό, σημαίνει ότι δεν υπάρχουν διπλότυπα.

SELECT 
  product_id, 
  count(*) AS occurrences
FROM bootcamp.bronze_marios_urban.brz_products
GROUP BY product_id
HAVING count(*) > 1;

In [0]:
%sql

--Βήμα 3.6: Έλεγχος για άκυρες τιμές, κόστη ή κενές κατηγορίες.

SELECT *
FROM bootcamp.bronze_marios_urban.brz_products
WHERE cast(list_price AS DECIMAL(10,2)) <= 0
   OR cast(unit_cost AS DECIMAL(10,2)) <= 0
   OR category IS NULL 
   OR trim(category) = '';

In [0]:
%sql
--Βήμα 3.7:Στέλνω στο etl_rejects τα προϊόντα που έχουν μη έγκυρη τιμή (<= 0) ή κενή κατηγορία/brand.Περιμένω 1 εγγραφή (P9999).


INSERT INTO bootcamp.silver_marios_urban.etl_rejects (source_table, reject_reason, rejected_record, rejected_at)
SELECT 
  'brz_products' AS source_table,
  'INVALID_PRICE_OR_CATEGORY' AS reject_reason,
  concat_ws(', ', product_id, product_name, category, subcategory, brand, list_price, unit_cost) AS rejected_record,
  current_timestamp() AS rejected_at
FROM bootcamp.bronze_marios_urban.brz_products
WHERE cast(list_price AS DECIMAL(10,2)) <= 0 
   OR category IS NULL 
   OR trim(category) = '';

In [0]:
%sql
--Βήμα 3.8: Δημιουργία του slv_products:
-- Standardise text: trim() στα κείμενα.
-- Cast types: μετατροπή unit_cost και list_price σε DECIMAL(10,2), και του is_active σε BOOLEAN.
--Φιλτράρω τα άκυρα προϊόντα (έχουν πάει στο etl_rejects).
--Περιμένω 75 καθαρά προϊόντα (από τα 76 αρχικά).

CREATE OR REPLACE TABLE bootcamp.silver_marios_urban.slv_products AS
SELECT 
  trim(product_id) AS product_id,
  trim(product_name) AS product_name,
  trim(category) AS category,
  trim(subcategory) AS subcategory,
  trim(brand) AS brand,
  cast(unit_cost AS DECIMAL(10,2)) AS unit_cost,
  cast(list_price AS DECIMAL(10,2)) AS list_price,
  CASE WHEN upper(trim(is_active)) = 'Y' THEN true ELSE false END AS is_active,
  current_timestamp() AS _processed_at
FROM bootcamp.bronze_marios_urban.brz_products
WHERE cast(list_price AS DECIMAL(10,2)) > 0 
  AND category IS NOT NULL 
  AND trim(category) != '';

-- Έλεγχος: Περιμένω 75 καθαρά προϊόντα
SELECT count(*) AS total_products FROM bootcamp.silver_marios_urban.slv_products;

In [0]:
%sql
--Βήμα 3.9:Έλεγχος για brz_stores:
--1) Έλεγχος για διπλότυπα store_id.
--2) Έλεγχος για NULL τιμές.

SELECT 
  store_id, 
  count(*) AS occurrences
FROM bootcamp.bronze_marios_urban.brz_stores
GROUP BY store_id
HAVING count(*) > 1;

SELECT *
FROM bootcamp.bronze_marios_urban.brz_stores
WHERE store_id IS NULL 
   OR store_name IS NULL 
   OR open_date IS NULL;

In [0]:
%sql
--Βήμα 3.10: Δημιουργία του slv_stores:
-- Standardise text: trim() στα ονόματα, τύπους και τοποθεσίες καταστημάτων.
-- Cast types: μετατροπή open_date σε DATE και size_sqm σε INT.
-- Δεν υπάρχουν σφάλματα/rejects σε αυτό το αρχείο.

CREATE OR REPLACE TABLE bootcamp.silver_marios_urban.slv_stores AS
SELECT 
  trim(store_id) AS store_id,
  trim(store_name) AS store_name,
  trim(store_type) AS store_type,
  trim(city) AS city,
  trim(region) AS region,
  cast(open_date AS DATE) AS open_date,
  cast(size_sqm AS INT) AS size_sqm,
  current_timestamp() AS _processed_at
FROM bootcamp.bronze_marios_urban.brz_stores;

-- Έλεγχος: Περιμένω 20 καταστήματα
SELECT count(*) AS total_stores FROM bootcamp.silver_marios_urban.slv_stores;

In [0]:
%sql
--Βήμα 3.11: Έλεγχοι ποιότητας στο brz_orders
--1) Έλεγχος για διπλότυπα order_id.
--2) Έλεγχος για NULL customer_id ή store_id.
--3) Έλεγχος για άκυρες μορφές ημερομηνιών (try_to_timestamp).
-- 1. Διπλότυπα order_id
SELECT order_id, count(*) 
FROM bootcamp.bronze_marios_urban.brz_orders 
GROUP BY order_id 
HAVING count(*) > 1;






In [0]:
%sql
-- 2. Κενά customer_id ή store_id
SELECT count(*) AS missing_customers 
FROM bootcamp.bronze_marios_urban.brz_orders 
WHERE customer_id IS NULL OR trim(customer_id) = '';

In [0]:
%sql
-- 3. Μη έγκυρα timestamps
SELECT count(*) AS invalid_timestamps 
FROM bootcamp.bronze_marios_urban.brz_orders 
WHERE try_to_timestamp(order_ts) IS NULL;

In [0]:
%sql
/*
  Οι 312 λανθασμενες εγγραφες και τα 76 invalid_timestamps μου φανηκαν παρα πολλα
  Διερεύνηση:
  1) Πώς μοιάζουν τα 312 κενά customer_id;
  2) Ποιο ακριβώς format έχουν τα 76 timestamps που δεν διαβάζονται;
*/

-- Δείγμα από τα κενά customer_id
SELECT * 
FROM bootcamp.bronze_marios_urban.brz_orders 
WHERE customer_id IS NULL OR trim(customer_id) = ''
LIMIT 5;



In [0]:
%sql
-- Δείγμα από τα 76 timestamps που βγαίνουν NULL
SELECT order_id, order_ts, customer_id
FROM bootcamp.bronze_marios_urban.brz_orders 
WHERE try_to_timestamp(order_ts) IS NULL
LIMIT 5;

In [0]:
%sql
/*
  Έλεγχος parsing ημερομηνιών:
  Δοκιμάζω standard ISO και dd/MM/yyyy HH:mm.
  Αν invalid_count = 0, διαβάζονται όλες επιτυχώς
*/
SELECT 
  count(*) AS total_orders,
  count_if(
    coalesce(
      try_to_timestamp(order_ts),
      try_to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')
    ) IS NULL
  ) AS invalid_timestamps
FROM bootcamp.bronze_marios_urban.brz_orders;

In [0]:
%sql

--Βήμα 3.12: Δημιουργία του slv_orders:
-- Standardise text: trim() στα πεδία κειμένου.
-- Parse dates: χρήση coalesce() με try_to_timestamp() για υποστήριξη τόσο του ISO format όσο και του dd/MM/yyyy HH:mm.
-- Κρατάμε και τις παραγγελίες με κενό customer_id (θα πάρουν key -1 στο Gold), τις χρειαζομαι γιατι διαφορετικα θα αλλοιωθουν τα τελικα μου αποτελεσματα στο reporting.
-- Περιμένω ακριβώς 8.000 παραγγελίες.

CREATE OR REPLACE TABLE bootcamp.silver_marios_urban.slv_orders AS
SELECT 
  trim(order_id) AS order_id,
  coalesce(
    try_to_timestamp(order_ts),
    try_to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')
  ) AS order_ts,
  trim(customer_id) AS customer_id,
  trim(store_id) AS store_id,
  trim(payment_method) AS payment_method,
  trim(order_status) AS order_status,
  current_timestamp() AS _processed_at
FROM bootcamp.bronze_marios_urban.brz_orders;

-- Έλεγχος: Περιμένω 8000 εγγραφές
SELECT count(*) AS total_orders FROM bootcamp.silver_marios_urban.slv_orders;

In [0]:
%sql
--Βήμα 3.13: Έλεγχος ποιότητας στο brz_order_items
--total_rows: συνολικές γραμμές
--duplicate_items: διπλότυπα order_item_id
-- invalid_quantities: ποσότητες <= 0
-- invalid_prices: τιμές μονάδας <= 0
-- invalid_discounts: εκπτώσεις εκτός ορίων [0, 1]

SELECT 
  count(*) AS total_rows,
  count(*) - count(DISTINCT order_line_id) AS duplicate_items,
  count_if(cast(quantity AS INT) <= 0) AS invalid_quantities,
  count_if(cast(unit_price AS DECIMAL(10,2)) <= 0) AS invalid_prices,
  count_if(cast(discount_pct AS DECIMAL(5,2)) < 0 OR cast(discount_pct AS DECIMAL(5,2)) > 1) AS invalid_discounts
FROM bootcamp.bronze_marios_urban.brz_order_items;

In [0]:
%sql
--Βήμα 3.14: Στέλνω στο etl_rejects τις γραμμές με ποσότητα <= 0, περιμένω ακριβώς 113 εγγραφές.

INSERT INTO bootcamp.silver_marios_urban.etl_rejects (source_table, reject_reason, rejected_record, rejected_at)
SELECT 
  'brz_order_items' AS source_table,
  'INVALID_QUANTITY' AS reject_reason,
  concat_ws(', ', order_line_id, order_id, product_id, quantity, unit_price, discount_pct) AS rejected_record,
  current_timestamp() AS rejected_at
FROM bootcamp.bronze_marios_urban.brz_order_items
WHERE cast(quantity AS INT) <= 0;

In [0]:
%sql
--Βήμα 3.14:Δημιουργία του slv_order_items:
-- Standardise text: trim() στα IDs.
-- Cast types: quantity σε INT, unit_price και discount_pct σε DECIMAL.
-- Υπολογισμός derived measures: gross_amount, discount_amount, net_amount.
-- Φιλτράρισμα: quantity > 0 (οι 113 απορρίψεις πήγαν στο etl_rejects).
-- Περιμένω ακριβώς 23.998 καθαρές γραμμές.

CREATE OR REPLACE TABLE bootcamp.silver_marios_urban.slv_order_items AS
WITH clean_items AS (
  SELECT 
    trim(order_line_id) AS order_line_id,
    trim(order_id) AS order_id,
    trim(product_id) AS product_id,
    cast(quantity AS INT) AS quantity,
    cast(unit_price AS DECIMAL(10,2)) AS unit_price,
    cast(discount_pct AS DECIMAL(5,2)) AS discount_pct
  FROM bootcamp.bronze_marios_urban.brz_order_items
  WHERE cast(quantity AS INT) > 0
)
SELECT 
  order_line_id,
  order_id,
  product_id,
  quantity,
  unit_price,
  discount_pct,
  round(quantity * unit_price, 2) AS gross_amount,
  round(round(quantity * unit_price, 2) * discount_pct, 2) AS discount_amount,
  round(round(quantity * unit_price, 2) - round(round(quantity * unit_price, 2) * discount_pct, 2), 2) AS net_amount,
  current_timestamp() AS _processed_at
FROM clean_items;

-- Έλεγχος: Περιμένω 23998 γραμμές (24111 - 113)
SELECT count(*) AS total_order_items FROM bootcamp.silver_marios_urban.slv_order_items;

In [0]:
%sql
--Βήμα 3.15:Συνοπτικός έλεγχος απορρίψεων ανά πίνακα και αιτία:
-- brz_customers: 15 (κενά email)
-- brz_products: 1 (άκυρη τιμή/κατηγορία)
-- brz_order_items: 113 (άκυρη ποσότητα <= 0)
--Σύνολο: 129 απορρίψεις

SELECT 
  source_table, 
  reject_reason, 
  count(*) AS total_rejected
FROM bootcamp.silver_marios_urban.etl_rejects
GROUP BY source_table, reject_reason
ORDER BY source_table;